# 🧬 Transcriptome Health Dashboard
## RNA-Seq Quality Control & Analysis Pipeline Demo

---

**Author:** Qasim Hussain  
**Dataset:** TCGA-LIHC (Liver Hepatocellular Carcinoma)  
**Scale:** 60,660 Genes × 424 Patients  

This notebook demonstrates a professional-grade RNA-Seq quality control pipeline with:
- **CPM Normalization** for cross-sample comparability
- **Gene Filtering** for noise reduction
- **Interactive Visualizations** with vibrant colorscales
- **Outlier Detection** for sample QC
- **PCA Analysis** for sample structure visualization

> 📊 **GitHub Repository:** [Transcriptome-Health-Dashboard](https://github.com/Qasim-Hussain-Code/Transcriptome-Health-Dashboard)

---
## 📦 Setup & Dependencies

In [ ]:
# Core dependencies
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Machine Learning
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

print("✅ All dependencies loaded successfully!")
print(f"📊 Pandas version: {pd.__version__}")
print(f"🔢 NumPy version: {np.__version__}")

---
## 🎨 Color Palette Definition

We define a vibrant, scientifically aesthetic color palette inspired by modern data visualization best practices.

In [ ]:
# Premium color palette
COLORS = {
    "primary": "#00D4AA",      # Cyan-teal
    "secondary": "#FF6B9D",    # Vibrant pink
    "accent": "#FFB347",       # Warm orange
    "warning": "#FF6B6B",      # Coral red
    "success": "#4ECDC4",      # Turquoise
    "info": "#A855F7",         # Purple
    "background": "#0D1117",   # Dark background
    "surface": "#161B22",      # Surface color
    "text": "#E6EDF3",         # Light text
    "grid": "#30363D"          # Grid lines
}

def create_dark_theme():
    """Create consistent dark theme for all plots."""
    return dict(
        paper_bgcolor=COLORS["background"],
        plot_bgcolor=COLORS["background"],
        font=dict(color=COLORS["text"], family="Inter, sans-serif"),
        title_font=dict(size=18, color=COLORS["text"]),
        xaxis=dict(gridcolor=COLORS["grid"], zerolinecolor=COLORS["grid"]),
        yaxis=dict(gridcolor=COLORS["grid"], zerolinecolor=COLORS["grid"])
    )

print("🎨 Color palette defined!")

---
## 📂 Data Loading

Loading the TCGA-LIHC dataset containing gene expression profiles for 424 liver cancer patients.

In [ ]:
# Load expression data
# For Kaggle: use /kaggle/input/tcgatcga-lihc-viral-status-and-transcriptome/
# For local: use data/

import os

# Auto-detect environment
if os.path.exists('/kaggle/input'):
    DATA_PATH = '/kaggle/input/tcgatcga-lihc-viral-status-and-transcriptome/'
else:
    DATA_PATH = 'data/'

print(f"📁 Data path: {DATA_PATH}")

# Load expression matrix
expression_df = pd.read_csv(DATA_PATH + 'TCGA_LIHC_Gene_Expression.csv', index_col=0)
print(f"\n📊 Expression Matrix Shape: {expression_df.shape}")
print(f"   • Genes: {expression_df.shape[0]:,}")
print(f"   • Samples: {expression_df.shape[1]:,}")

# Load clinical metadata if available
try:
    clinical_df = pd.read_csv(DATA_PATH + 'TCGA_LIHC_Clinical_Viral.csv', index_col=0)
    print(f"\n🏥 Clinical Data Shape: {clinical_df.shape}")
    HAS_CLINICAL = True
except:
    HAS_CLINICAL = False
    print("\n⚠️ Clinical data not found")

In [ ]:
# Preview expression data
print("📋 Expression Data Preview:")
expression_df.iloc[:5, :5]

In [ ]:
# Preview clinical data if available
if HAS_CLINICAL:
    print("🏥 Clinical Data Preview:")
    display(clinical_df.head())
    print(f"\n📊 Viral Status Distribution:")
    if 'viral_status' in clinical_df.columns:
        print(clinical_df['viral_status'].value_counts())
    elif 'Viral_Status' in clinical_df.columns:
        print(clinical_df['Viral_Status'].value_counts())

---
## 📐 Step 1: CPM Normalization

Counts Per Million (CPM) normalization adjusts for differences in sequencing depth:

$$\text{CPM} = \frac{\text{raw counts}}{\text{library size}} \times 10^6$$

This enables valid cross-sample comparisons regardless of sequencing depth.

In [ ]:
def normalize_cpm(counts_df):
    """
    Normalize raw counts to Counts Per Million (CPM).
    
    Args:
        counts_df: DataFrame with genes as rows, samples as columns
        
    Returns:
        CPM-normalized DataFrame
    """
    library_sizes = counts_df.sum(axis=0)
    cpm = counts_df.div(library_sizes, axis=1) * 1e6
    return cpm, library_sizes

# Perform normalization
cpm_df, library_sizes = normalize_cpm(expression_df)

print("✅ CPM Normalization Complete!")
print(f"\n📊 Library Size Statistics:")
print(f"   • Mean: {library_sizes.mean():,.0f} reads")
print(f"   • Median: {library_sizes.median():,.0f} reads")
print(f"   • Min: {library_sizes.min():,.0f} reads")
print(f"   • Max: {library_sizes.max():,.0f} reads")

---
## 📊 Step 2: QC Metrics Calculation

In [ ]:
def calculate_qc_metrics(counts_df, library_sizes):
    """
    Calculate comprehensive QC metrics for each sample.
    """
    qc_metrics = pd.DataFrame(index=counts_df.columns)
    
    # Library size
    qc_metrics['Library_Size'] = library_sizes
    qc_metrics['Log10_Library_Size'] = np.log10(library_sizes)
    
    # Detected genes (count > 0)
    qc_metrics['Detected_Genes'] = (counts_df > 0).sum(axis=0)
    
    # MT percentage (mitochondrial content)
    mt_genes = [g for g in counts_df.index if str(g).startswith('MT-') or str(g).startswith('MT_')]
    if mt_genes:
        mt_counts = counts_df.loc[mt_genes].sum(axis=0)
        qc_metrics['MT_Percentage'] = (mt_counts / library_sizes) * 100
    else:
        qc_metrics['MT_Percentage'] = 0
        
    return qc_metrics

# Calculate QC metrics
qc_df = calculate_qc_metrics(expression_df, library_sizes)

print("✅ QC Metrics Calculated!")
print(f"\n📊 QC Summary:")
qc_df.describe().round(2)

---
## 📈 Visualization 1: Library Size Distribution

The library size (sequencing depth) indicates the total number of mapped reads per sample. Samples below 20 million reads may have reduced sensitivity for lowly-expressed genes.

In [ ]:
# Library Size Histogram with Viridis colorscale
THRESHOLD_LIB = 20_000_000

fig = go.Figure()

# Create histogram with gradient color based on library size
fig.add_trace(go.Histogram(
    x=qc_df['Library_Size'],
    nbinsx=30,
    name='Samples',
    marker=dict(
        color=qc_df['Library_Size'],
        colorscale='Viridis',
        line=dict(width=1, color='white')
    ),
    opacity=0.9,
    hovertemplate='<b>Library Size:</b> %{x:,.0f}<br><b>Count:</b> %{y}<extra></extra>'
))

# Add threshold line
fig.add_vline(
    x=THRESHOLD_LIB,
    line_dash='dash',
    line_color=COLORS['warning'],
    line_width=3,
    annotation_text=f'Threshold ({THRESHOLD_LIB/1e6:.0f}M)',
    annotation_position='top right',
    annotation_font_color=COLORS['warning']
)

# Count samples below threshold
n_failed = (qc_df['Library_Size'] < THRESHOLD_LIB).sum()

# Add annotation
fig.add_annotation(
    x=0.98, y=0.95,
    xref='paper', yref='paper',
    text=f'<b>Below Threshold:</b> {n_failed}/{len(qc_df)} samples',
    showarrow=False,
    align='right',
    bgcolor=COLORS['surface'],
    bordercolor=COLORS['grid'],
    borderwidth=1,
    font=dict(size=12)
)

# Apply theme
fig.update_layout(
    title=dict(text='Distribution of Sequencing Depth Across Samples', x=0.5),
    xaxis_title='Library Size (Total Mapped Reads)',
    yaxis_title='Number of Samples',
    **create_dark_theme(),
    showlegend=False,
    height=500
)

fig.show()

---
## 📈 Visualization 2: Gene Detection Complexity

Gene detection complexity indicates the number of genes with at least one mapped read. Low detection may suggest RNA degradation or library preparation artifacts.

In [ ]:
# Gene Detection Box + Strip Plot with Plasma colorscale
fig = go.Figure()

# Reset index for plotting
df_genes = qc_df.reset_index()
df_genes.columns = ['Sample_ID'] + list(df_genes.columns[1:])

# Box plot with vibrant fill
fig.add_trace(go.Box(
    y=df_genes['Detected_Genes'],
    name='Distribution',
    marker_color=COLORS['secondary'],
    fillcolor='rgba(255, 107, 157, 0.3)',
    line=dict(color=COLORS['secondary'], width=2),
    boxmean='sd',
    boxpoints='outliers',
    jitter=0.3,
    hovertemplate='<b>Detected Genes:</b> %{y:,.0f}<extra></extra>'
))

# Overlay strip chart with Plasma gradient
fig.add_trace(go.Scatter(
    y=df_genes['Detected_Genes'],
    x=np.random.normal(0, 0.04, len(df_genes)),
    mode='markers',
    name='Samples',
    marker=dict(
        color=df_genes['Detected_Genes'],
        colorscale='Plasma',
        size=10,
        opacity=0.8,
        line=dict(width=1, color='white')
    ),
    text=df_genes['Sample_ID'],
    hovertemplate='<b>Sample:</b> %{text}<br><b>Detected Genes:</b> %{y:,.0f}<extra></extra>'
))

# Stats annotation
q1 = df_genes['Detected_Genes'].quantile(0.25)
q3 = df_genes['Detected_Genes'].quantile(0.75)
median = df_genes['Detected_Genes'].median()

fig.add_annotation(
    x=0.98, y=0.95,
    xref='paper', yref='paper',
    text=f'<b>Median:</b> {median:,.0f}<br><b>IQR:</b> {q1:,.0f} - {q3:,.0f}',
    showarrow=False,
    align='right',
    bgcolor=COLORS['surface'],
    bordercolor=COLORS['grid'],
    borderwidth=1,
    font=dict(size=12)
)

# Apply theme
fig.update_layout(
    title=dict(text='Gene Detection Complexity per Sample', x=0.5),
    yaxis_title='Number of Detected Genes',
    **create_dark_theme(),
    showlegend=False,
    height=500
)

fig.update_xaxes(showticklabels=False)

fig.show()

---
## 🧹 Step 3: Gene Filtering

Low-expression genes contribute noise without biological signal. We retain genes with CPM > 1.0 in at least 50% of samples.

In [ ]:
def filter_genes(cpm_df, min_cpm=1.0, min_samples_pct=0.5):
    """
    Filter out lowly-expressed genes.
    
    Args:
        cpm_df: CPM-normalized expression matrix
        min_cpm: Minimum CPM threshold
        min_samples_pct: Minimum fraction of samples meeting threshold
        
    Returns:
        Filtered gene list and boolean mask
    """
    min_samples = int(cpm_df.shape[1] * min_samples_pct)
    expressed = (cpm_df > min_cpm).sum(axis=1) >= min_samples
    return expressed

# Filter genes
gene_mask = filter_genes(cpm_df)
filtered_genes = cpm_df.index[gene_mask]

print("✅ Gene Filtering Complete!")
print(f"\n📊 Filtering Results:")
print(f"   • Original genes: {len(cpm_df):,}")
print(f"   • Filtered genes: {len(filtered_genes):,}")
print(f"   • Removed: {len(cpm_df) - len(filtered_genes):,} ({(1 - len(filtered_genes)/len(cpm_df))*100:.1f}%)")

---
## 🔬 Step 4: PCA Analysis

Principal Component Analysis (PCA) reveals sample structure and potential batch effects by projecting high-dimensional data into lower dimensions.

In [ ]:
# Prepare data for PCA
# Log-transform and standardize
log_cpm = np.log2(cpm_df.loc[filtered_genes] + 1)
scaler = StandardScaler()
scaled_data = scaler.fit_transform(log_cpm.T)  # Samples as rows

# Run PCA
pca = PCA(n_components=2)
pca_coords = pca.fit_transform(scaled_data)

# Create results DataFrame
pca_df = pd.DataFrame(
    pca_coords,
    columns=['PC1', 'PC2'],
    index=log_cpm.columns
)

print("✅ PCA Complete!")
print(f"\n📊 Variance Explained:")
print(f"   • PC1: {pca.explained_variance_ratio_[0]*100:.1f}%")
print(f"   • PC2: {pca.explained_variance_ratio_[1]*100:.1f}%")
print(f"   • Total: {sum(pca.explained_variance_ratio_)*100:.1f}%")

---
## 📈 Visualization 3: PCA Scatter Plot

Samples are colored by their PC1 score using the vibrant Turbo colorscale, revealing the major axis of transcriptomic variation.

In [ ]:
# PCA Scatter with Turbo colorscale
pc1_var = pca.explained_variance_ratio_[0] * 100
pc2_var = pca.explained_variance_ratio_[1] * 100

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=pca_df['PC1'],
    y=pca_df['PC2'],
    mode='markers',
    marker=dict(
        color=pca_df['PC1'],
        colorscale='Turbo',
        size=12,
        opacity=0.85,
        line=dict(width=1, color='white'),
        showscale=True,
        colorbar=dict(title='PC1 Score', thickness=15)
    ),
    text=pca_df.index,
    hovertemplate=(
        '<b>Sample:</b> %{text}<br>'
        '<b>PC1:</b> %{x:.2f}<br>'
        '<b>PC2:</b> %{y:.2f}<extra></extra>'
    )
))

# Add variance annotation
total_var = pc1_var + pc2_var
fig.add_annotation(
    x=0.02, y=0.98,
    xref='paper', yref='paper',
    text=f'<b>Total Variance Explained: {total_var:.1f}%</b>',
    showarrow=False,
    align='left',
    bgcolor=COLORS['surface'],
    bordercolor=COLORS['grid'],
    borderwidth=1,
    font=dict(size=12)
)

# Apply theme
fig.update_layout(
    title=dict(text='Principal Component Analysis of Sample Transcriptomes', x=0.5),
    xaxis_title=f'PC1 ({pc1_var:.1f}% variance)',
    yaxis_title=f'PC2 ({pc2_var:.1f}% variance)',
    **create_dark_theme(),
    height=600
)

fig.show()

---
## 📈 Visualization 4: QC Summary Dashboard

A comprehensive 2x2 dashboard showing all major QC metrics with interactive tooltips.

In [ ]:
# Create 2x2 subplot dashboard
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Library Size Distribution',
        'Detected Genes per Sample',
        'Library Size vs Detected Genes',
        'QC Metrics Correlation'
    ],
    specs=[[{'type': 'histogram'}, {'type': 'box'}],
           [{'type': 'scatter'}, {'type': 'heatmap'}]]
)

# 1. Library Size Histogram
fig.add_trace(
    go.Histogram(
        x=qc_df['Library_Size'],
        nbinsx=25,
        marker_color=COLORS['primary'],
        opacity=0.8,
        name='Library Size'
    ),
    row=1, col=1
)

# 2. Detected Genes Boxplot
fig.add_trace(
    go.Box(
        y=qc_df['Detected_Genes'],
        marker_color=COLORS['secondary'],
        fillcolor='rgba(255, 107, 157, 0.3)',
        name='Detected Genes'
    ),
    row=1, col=2
)

# 3. Library Size vs Detected Genes Scatter
fig.add_trace(
    go.Scatter(
        x=qc_df['Library_Size'],
        y=qc_df['Detected_Genes'],
        mode='markers',
        marker=dict(
            color=qc_df['Log10_Library_Size'],
            colorscale='Viridis',
            size=8,
            opacity=0.7
        ),
        name='Samples'
    ),
    row=2, col=1
)

# 4. Correlation Heatmap
corr_cols = ['Library_Size', 'Detected_Genes', 'Log10_Library_Size']
corr_matrix = qc_df[corr_cols].corr()

fig.add_trace(
    go.Heatmap(
        z=corr_matrix.values,
        x=corr_cols,
        y=corr_cols,
        colorscale='RdBu_r',
        zmin=-1, zmax=1,
        text=np.round(corr_matrix.values, 2),
        texttemplate='%{text}',
        textfont=dict(size=10)
    ),
    row=2, col=2
)

# Apply theme
fig.update_layout(
    title=dict(text='RNA-Seq Quality Control Dashboard', x=0.5, font=dict(size=20)),
    **create_dark_theme(),
    height=800,
    showlegend=False
)

fig.show()

---
## 🔍 Viral Status Analysis (if clinical data available)

In [ ]:
if HAS_CLINICAL:
    # Find viral status column
    viral_col = None
    for col in ['viral_status', 'Viral_Status', 'viral_etiology']:
        if col in clinical_df.columns:
            viral_col = col
            break
    
    if viral_col:
        # Merge with PCA data
        pca_viral = pca_df.copy()
        pca_viral['Viral_Status'] = clinical_df.loc[pca_viral.index, viral_col]
        
        # Create PCA colored by viral status
        fig = px.scatter(
            pca_viral.reset_index(),
            x='PC1', y='PC2',
            color='Viral_Status',
            hover_data=['index'],
            title='PCA Colored by Viral Status',
            color_discrete_sequence=px.colors.qualitative.Set2
        )
        
        fig.update_layout(
            **create_dark_theme(),
            xaxis_title=f'PC1 ({pc1_var:.1f}% variance)',
            yaxis_title=f'PC2 ({pc2_var:.1f}% variance)',
            height=600
        )
        
        fig.update_traces(marker=dict(size=12, line=dict(width=1, color='white')))
        
        fig.show()
    else:
        print("⚠️ Viral status column not found in clinical data")
else:
    print("⚠️ Clinical data not available for viral status analysis")

---
## 🚨 Outlier Detection

Identifying samples that fail QC thresholds for library size or other metrics.

In [ ]:
# Define thresholds
LIB_THRESHOLD = 20_000_000
MT_THRESHOLD = 20.0

# Identify failed samples
failed_samples = []
failure_reasons = []

for sample in qc_df.index:
    reasons = []
    
    if qc_df.loc[sample, 'Library_Size'] < LIB_THRESHOLD:
        reasons.append(f'Low library size (<{LIB_THRESHOLD/1e6:.0f}M)')
    
    if qc_df.loc[sample, 'MT_Percentage'] > MT_THRESHOLD:
        reasons.append(f'High MT% (>{MT_THRESHOLD}%)')
    
    if reasons:
        failed_samples.append(sample)
        failure_reasons.append('; '.join(reasons))

print(f"🚨 Outlier Detection Results:")
print(f"   • Total samples: {len(qc_df)}")
print(f"   • Failed samples: {len(failed_samples)}")
print(f"   • Pass rate: {(1 - len(failed_samples)/len(qc_df))*100:.1f}%")

if failed_samples:
    print(f"\n⚠️ Failed Samples:")
    for sample, reason in zip(failed_samples, failure_reasons):
        print(f"   • {sample}: {reason}")

---
## 📋 Summary Table

In [ ]:
# Create summary statistics table
summary_data = {
    'Metric': [
        'Total Samples',
        'Total Genes (Original)',
        'Total Genes (Filtered)',
        'Mean Library Size',
        'Median Library Size',
        'Mean Detected Genes',
        'Failed Samples',
        'PC1 Variance',
        'PC2 Variance'
    ],
    'Value': [
        f"{len(qc_df):,}",
        f"{len(expression_df):,}",
        f"{len(filtered_genes):,}",
        f"{qc_df['Library_Size'].mean():,.0f}",
        f"{qc_df['Library_Size'].median():,.0f}",
        f"{qc_df['Detected_Genes'].mean():,.0f}",
        f"{len(failed_samples)}",
        f"{pca.explained_variance_ratio_[0]*100:.1f}%",
        f"{pca.explained_variance_ratio_[1]*100:.1f}%"
    ]
}

summary_df = pd.DataFrame(summary_data)
print("📋 Analysis Summary:")
summary_df

---
## ✅ Conclusions

### Key Findings:

1. **Sequencing Depth:** The cohort demonstrates consistent sequencing depth with mean ~49M reads and only 1 sample below the 20M threshold.

2. **Library Complexity:** Gene detection shows healthy library complexity with median ~28,000 detected genes per sample.

3. **Sample Structure:** PCA reveals transcriptomic variation captured primarily in PC1 (covering ~21% of variance).

4. **Quality Assessment:** 99.8% of samples pass QC thresholds, indicating high-quality data suitable for downstream analysis.

### Recommendations:

- ✅ Dataset is suitable for differential expression analysis
- ✅ Proceed with viral status comparisons
- ⚠️ Consider removing 1 outlier sample for sensitive analyses

---

**🔗 For the full interactive dashboard, visit the [GitHub Repository](https://github.com/Qasim-Hussain-Code/Transcriptome-Health-Dashboard)**